# Create Alberta Innovates Awards

Creates Alberta Innovates awards from the agency's own funded-projects pages
(https://albertainnovates.ca/impact/funded-projects/; WordPress post type
`project`, one page per project at /projects/{slug}/). Each page has a title,
a description and a "Project at a Glance" block: Funding Recipient, Funding
Awarded (CAD), Project Duration, Sector, Program, Related Files.
**899 awards, 2016-2026, CAD 206.5M** (local run 2026-10-01): 594 project pages
plus 305 rows from cohort tables.

**Which posts are awards (filter, reviewed 2026-10-01):**
- 23 posts are *cohort* pages (AICE Concepts / Validate / Market Access, LevMax-Health 1.0-5.0,
  PRIHS 5-7, AI-Better Health, Postdoctoral Fellowships, DICE Open Calls, Bitumen Beyond Combustion
  2018, Ecosystem Development Partnerships Rounds 1-2) whose table lists one row per funded project.
  Each table row is one award (`record_type = 'cohort_table_row'`); the cohort page itself is not.
  Table rows that also have their own project page are dropped in favour of the page (15); the
  table-only document index 'Bitumen Partial Upgrading - Funded Projects' (links to project pages) is skipped.
- Programme reports/summaries with no single recipient and no table (9 pages: "Carbon Storage -
  A Summary of Experience...", "Hydrogen Centre of Excellence Public Update Report", "Care in the
  Community 2018/2019", ...) are dropped; WordPress duplicate posts ("-2" slugs, same title, amount
  and dates) are kept once (8).
- Every programme is kept, including training (Postdoctoral Fellowships -> `funding_type =
  'fellowship'`) and the Ecosystem Development Partnerships Program (45 rows of innovation-ecosystem
  support: hubs, accelerators, training programmes) under the 2026-10-01 scope rule (keep and flag).

**Prerequisites:**
- Run `scripts/local/alberta_innovates_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/alberta_innovates/alberta_innovates_projects.parquet`
  (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** citing works write Alberta Innovates' 9-digit
application number ("Advance 232403381", "CASBE 222301211", "202102818"; checked against
`openalex_awards_raw` citation stubs for F4320325651, 2026-10-01). The pages do not print it as a
field, but the agency's own Related-Files PDFs are named after it
(`.../232404951-Bayat-Project-Summary-WIP.pdf`). Three forms, recorded in `award_id_source`:
- `application_number` (104): one 9-digit number leads the page's related-file names;
- `slug` (490): otherwise the project URL slug, the public-facing reference;
- `cohort_slug` (305): cohort-table rows, `{cohort-page-slug}/{project-title-slug}`.

**Amounts / dates:** CAD (single-country funder). Free-text amounts ("Total value ... $8.8 million,
with $7.2 million contributed from Alberta Innovates") resolve to the Alberta Innovates share.
Cohort rows take the per-row Value/Award Date when the table has them, else the cohort page's own
duration; when neither exists `start_year` comes from the cohort year in the page title
("LevMax 1.0 (2022)", "PRIHS 5 (2019)") and `start_date` stays NULL.

**lead_investigator:** "Funding Recipient" is usually an organization (company, university,
association). When it names a person ("University of Alberta, Dr. Alireza Bayat"; cohort "Lead
Applicant" cells) the person is the lead and the organization the affiliation; otherwise the lead
carries only the recipient organization as affiliation (names NULL), HHMI/Mott precedent.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320325651`
- display_name: Alberta Innovates
- ror_id / doi: from `openalex.funders.funders` (no ROR; doi 10.13039/501100009192).
  The predecessor corporations (Alberta Innovates - Health Solutions F4320319927,
  - Technology Futures F4320319907, Bio Solutions F4320324027) are separate
  funders and are not mapped here: every project on the current site is shown as
  an Alberta Innovates award.

**Priority:** `483` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alberta_innovates_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/alberta_innovates/alberta_innovates_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_rows, COUNT_IF(record_type = 'cohort_table_row') as cohort_rows
FROM openalex.awards.alberta_innovates_raw;

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.alberta_innovates_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.alberta_innovates_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320325651 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320325651;

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320325651 must have exactly one row in openalex.funders.funders') AS funder_ok
FROM openalex.funders.funders
WHERE funder_id = 4320325651;

## Step 2: Create Alberta Innovates Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.alberta_innovates_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320325651  -- Alberta Innovates
),

lead AS (
    -- Named person when the recipient / lead applicant names one, else organization-only.
    -- Names were split in Python (split_name, runbook §2.4.1).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL
                  OR NULLIF(TRIM(recipient_organization), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(recipient_organization), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead_investigator
    FROM openalex.awards.alberta_innovates_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'CAD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.program = 'Postdoctoral Fellowships' THEN 'fellowship' ELSE 'grant' END as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'alberta_innovates_projects' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    COALESCE(YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')), TRY_CAST(g.cohort_year AS INT)) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN l.lead_investigator IS NOT NULL THEN array(l.lead_investigator) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.alberta_innovates_raw g
JOIN lead l ON l.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): funder_award_id is the 9-digit application number, the project URL
-- slug, or '{cohort-slug}/{title-slug}' for cohort-table rows (never a WP post id); ids unique.
SELECT
    assert_true(COUNT_IF(NOT (funder_award_id RLIKE '^[0-9]{9}$'
                              OR funder_award_id RLIKE '^[a-z0-9%-]*[a-z][a-z0-9%-]*$'
                              OR funder_award_id RLIKE '^[a-z0-9%-]+/[a-z0-9-]+$')) = 0,
                'funder_award_id is not an application number, slug or cohort key') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok
FROM openalex.awards.alberta_innovates_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alberta_innovates_projects' AND priority = 483;

-- Insert into openalex_awards_raw with priority.
-- Priority 483: direct-from-funder ingest of Alberta Innovates' own funded-projects pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    483 as priority
FROM openalex.awards.alberta_innovates_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.alberta_innovates_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.alberta_innovates_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.alberta_innovates_awards
GROUP BY funder_scheme, funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.alberta_innovates_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: lead / recipient top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.alberta_innovates_awards
WHERE lead_investigator.family_name IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.alberta_innovates_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_recipient
FROM openalex.awards.alberta_innovates_awards;

In [ ]:
%sql
-- Existing citation shells for F4320325651 that these awards upgrade.
SELECT c.provenance, COUNT(*) as shells, COUNT(t.id) as matched_by_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.alberta_innovates_awards t ON t.id = c.id
WHERE c.funder_id = 4320325651 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'alberta_innovates_projects'
GROUP BY provenance, priority;